# Import sempy and Fabric libraries

In [2]:
import sempy
import sempy.fabric as fabric
import pandas

StatementMeta(, 5ea9789f-8978-45d5-846d-b8c924a72672, 6, Finished, Available, Finished)

## Import Parameters

In [1]:
%run 2. Parameters

StatementMeta(, ef97095f-aaf6-4457-9126-eb82718d272d, 5, Finished, Available, Finished, True)

## Setup Workspace and Dataset, then Run DAX Query


In [9]:
df = fabric.evaluate_dax(datasetId, dax_string=dax_query, workspace=workspaceId)
display(df)

StatementMeta(, 5ea9789f-8978-45d5-846d-b8c924a72672, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7e8a66cc-6f22-42a9-9524-fd34b65e540c)

## Clean Column Names and Save DataFrame as Spark Table

In [10]:
df.columns = [col.strip("[]") for col in df.columns]
spark_df = spark.createDataFrame(df)
table_name = f"calcDependency"
spark_df.write.saveAsTable(name=table_name, mode="overwrite")

StatementMeta(, 5ea9789f-8978-45d5-846d-b8c924a72672, 14, Finished, Available, Finished)

In [ ]:
from datetime import datetime

df = fabric.evaluate_dax(datasetId, dax_string=dax_query, workspace=workspaceId)

# Add timestamp column
df['TIMESTAMP'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Clean column names
df.columns = [col.strip("[]") for col in df.columns]

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Define table name
table_name = "calcDependency"

# Determine version number based on existing data
if spark.catalog.tableExists(table_name):
    # Read existing table to find the max version for this semantic model
    existing_df = spark.read.table(table_name)
    
    # Filter for current semantic model and get max version
    max_version = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()[0]['max_ver']
    
    # Increment version, or start at 1 if this model hasn't been added yet
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
from pyspark.sql.functions import lit
spark_df = spark_df.withColumn('DATABASE_VERSION', lit(f"{semantic_model_name} | V{next_version}"))

# Write using the determined mode
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

display(df)

# Display the Data

In [11]:
df = spark.sql(f"SELECT * FROM {lakehouse_name}.{table_schema}.calcdependency")
display(df)

StatementMeta(, 5ea9789f-8978-45d5-846d-b8c924a72672, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 11fb6bca-986e-4260-a757-55186a26ad6d)